# Where the Carry Bit Comes From

### Reading order as a cut, and the residual it does not pay for

**Dean Kulik**, QuHarmonics Research Group — ORCID 0009-0003-3128-8828
A-Mark9 / NEXUS · September 2026 · v3

---

## Abstract

The previous orbit left two things open on the adder rung of the interface bridge: a
conjectured link *memory = number of orbits of relations under the reader's step*, and a
proof of the $\binom n2$ column of the relation residual $b_1 = \binom n2 + (n-1)$.

This notebook closes both, one by refutation and one by proof.

**1. A closed form for the retained state.** The minimal state an $n$-bit adder must
retain, when its input positions are presented in an arbitrary order, depends only on the
*set* of positions read so far — never on the order used to reach that set — and it
factorises over the maximal runs of that set:

$$F(S)\;=\;\prod_{\text{runs }[l,r]\subseteq S} f(l,r),\qquad
f=\begin{cases}
1 & \text{run is all of } [0,n-1]\\
2 & \text{run touches } 0 \text{ only}\\
L+1 & \text{run touches } n-1 \text{ only}\\
2L+1 & \text{interior run, } L=r-l+1
\end{cases}$$

Verified against brute-force Myhill–Nerode on every reading order for $n\le5$ and random
orders up to $n=8$: zero mismatches.

**2. What the state is.** An interior run retains an *owed-output prefix* (which sum bits
it cannot yet emit) crossed with a *carry transfer* (const 0, const 1, or transparent).
Memory is an unpaid ledger times a carry mode. A bottom-anchored run owes nothing, because
its carry-in is known, and retains only the carry bit.

**3. LSB-first is the minimum, and for $n\ge3$ the unique one.** Exactly one reading order
out of $n!$ attains 2 states for $n\ge3$; at $n=2$ both orders do. Every other order is strictly worse, and the worst needs
$4\cdot3^{(n-3)/2}$ (odd $n$) or $20\cdot3^{(n-6)/2}$ (even $n\ge6$) states — about
$0.79n$ bits instead of one.

**4. The conjecture is false, structurally.** Orbits of relations under the reader's step
number $n$; the measured memory is one bit at every width. The two quantities cannot be
equal by any counting map: the residual is a *rank*, additive over pairs; the memory is a
*state count*, multiplicative over runs. Worse, $2L+1$ is odd, so the retained state of any
cut containing an interior run is not a set of bits at all. The carry *bit* is a property
of the bottom-anchored cut, not of addition.

**5. The $\binom n2$ column, lower bound.** Explicit witness cocycles
$\varphi_{ij}(x\to x+2^i)=[\,\text{bits } i..j-1 \text{ of } x \text{ all } 1\,]$ vanish on
every declared square and pair with the undeclared squares through a matrix that is
triangular in the right coordinates, giving $b_1\ge\binom n2+(n-1)$ for all $n$.

**6. The upper bound, from a closed form (v2).** With $c_{kl}=r_{kl}(2^l-2^k)$ and $m(x)$
the run of 1s in $x$ starting at bit $j+1$, every undeclared square satisfies

$$\bigl[r_{ij}(x)\bigr]\;=\;(-1)^{x_j}\Bigl(c_{ij}+\sum_{t=1}^{m(x)}
\bigl(c_{i,j+t}-c_{j,j+t}\bigr)\Bigr)\pmod{B_D}.$$

Checked on **every** undeclared square at each width to $n=8$, at three primes and exactly
over $\mathbb Q$. So the $c_{ij}$ span, the two bounds meet, and $b_1=\binom n2+(n-1)$ is a
theorem at every width checked rather than a nine-width numerical law.

**7. Two of the three lemmas, proved (v3).** L1 (transport below the seam) and the base case
$j=n-1$ are now theorems for all $n$, the second an exact chain identity. What is left of the
general-$n$ statement is a single membership: the seam sign is the undeclared carry identity
$A_j^2=A_{j+1}$, and L2 below the top reduces to
$\partial r_{i,j+1}(z)+T_j(z+2^i)-T_j(z)\in B_D$.

## 0. What was open

From the previous orbit, verbatim:

> **Not yet:** a theorem linking relation residuals to memory; a proof of the $C(n,2)$
> column; any account of how physics selects a predicate.

> **Next orbit.** The $n-1$ carry identities $A_i^2=A_{i+1}$ are one relation shifted by
> bit position, and a serial adder's time step is exactly that shift. If relations related
> by the reader's own step are paid for by the same retained state, memory would count
> relation *orbits* under that step, not relations. For the carry identities that gives one
> orbit, and one bit. The $C(n,2)$ pairwise classes are not related by a single step, so
> they are the real test.

The run that produced the state counts was executed but its predictor was never written
down. Everything below is rebuilt from the specification alone — no code carried over.

## 1. Library

Self-contained. Two independent engines: a brute-force Myhill–Nerode minimiser that knows
nothing about carries, and an arena/homology engine that knows nothing about readers.

In [1]:
import itertools, math, random, time
from functools import lru_cache
import numpy as np

P = 1000003                      # prime field; |G| is a 2-power, so char-0 behaviour

# ============================================================ READER: brute force
def brute_profile(n, order):
    """Myhill-Nerode state count after each prefix length, by explicit enumeration.

    Model: inputs (a,b) presented one POSITION at a time in `order`; outputs are the n
    sum bits of (a+b) mod 2^n; a sum bit is emitted as soon as no remaining input can
    change it; two prefixes are the same state iff every completion gives identical
    not-yet-emitted outputs."""
    pairs = [(0,0),(0,1),(1,0),(1,1)]
    prof = []
    for t in range(n+1):
        known, unknown = order[:t], order[t:]
        sigs = set()
        for kv in itertools.product(pairs, repeat=t):
            outs = []
            for uv in itertools.product(pairs, repeat=n-t):
                a = b = 0
                for p,(x,y) in zip(known,kv):   a |= x<<p; b |= y<<p
                for p,(x,y) in zip(unknown,uv): a |= x<<p; b |= y<<p
                s = (a+b) % (1<<n)
                outs.append(tuple((s>>i)&1 for i in range(n)))
            pending = tuple(i for i in range(n) if len({o[i] for o in outs}) > 1)
            sigs.add((pending, tuple(tuple(o[i] for i in pending) for o in outs)))
        prof.append(len(sigs))
    return prof

# ============================================================ READER: closed form
def runs_of(S, n):
    out = []
    for p in sorted(S):
        if out and out[-1][1] == p-1: out[-1][1] = p
        else:                          out.append([p,p])
    return [tuple(r) for r in out]

def F(S, n):
    v = 1
    for (lo,hi) in runs_of(S,n):
        L = hi-lo+1; bot, top = lo==0, hi==n-1
        v *= 1 if (bot and top) else 2 if bot else (L+1) if top else (2*L+1)
    return v

def pred_profile(n, order): return [F(set(order[:t]), n) for t in range(n+1)]

def named_orders(n):
    mid = (n-1)//2; mo, up, dn, turn = [mid], mid+1, mid-1, True
    while len(mo) < n:
        if   turn and up <= n-1: mo.append(up); up += 1
        elif dn >= 0:            mo.append(dn); dn -= 1
        elif up <= n-1:          mo.append(up); up += 1
        turn = not turn
    return {"low->high": list(range(n)), "high->low": list(range(n-1,-1,-1)),
            "even/odd": [i for i in range(n) if i%2==0]+[i for i in range(n) if i%2],
            "middle-out": mo}

# ============================================================ ARENA: Cayley complex
def arena(n):
    N = 1<<n; eidx, elist = {}, []
    for x in range(N):
        for i in range(n):
            y = (x + (1<<i)) % N; key = (min(x,y), max(x,y))
            if key not in eidx: eidx[key] = len(elist); elist.append((x,i,y))
    return N, eidx, elist

def sgn_edge(eidx, x, i, n):
    y = (x + (1<<i)) % (1<<n); k = eidx[(min(x,y), max(x,y))]
    return k, (1 if x < y else -1)

def disjoint(x, i, j, n):
    N = 1<<n
    return ((x ^ ((x+(1<<i))%N)) & (x ^ ((x+(1<<j))%N))) == 0

def dsquare(n, eidx, x, i, j):
    N = 1<<n; r = {}
    for (base,gen,s) in ((x,i,+1), ((x+(1<<i))%N,j,+1), (x,j,-1), ((x+(1<<j))%N,i,-1)):
        k,sg = sgn_edge(eidx, base, gen, n); r[k] = r.get(k,0) + s*sg
    return r

def dtriangle(n, eidx, x, i):
    N = 1<<n; r = {}
    for (base,gen,s) in ((x,i,+1), ((x+(1<<i))%N,i,+1), (x,i+1,-1)):
        k,sg = sgn_edge(eidx, base, gen, n); r[k] = r.get(k,0) + s*sg
    return r

def rank_modp(rows, ncols):
    if not rows: return 0
    M = np.zeros((len(rows), ncols), dtype=np.int64)
    for a,r in enumerate(rows):
        for c,v in r.items(): M[a,c] = v % P
    r = 0
    for c in range(ncols):
        nz = np.nonzero(M[r:,c])[0]
        if nz.size == 0: continue
        piv = r + nz[0]; M[[r,piv]] = M[[piv,r]]
        M[r] = (M[r] * pow(int(M[r,c]), P-2, P)) % P
        below = np.nonzero(M[r+1:,c])[0]
        if below.size:
            idx = below + r + 1
            M[idx] = (M[idx] - np.outer(M[idx,c], M[r])) % P
        r += 1
        if r == M.shape[0]: break
    return r

def b1(n, which):
    N, eidx, elist = arena(n); E, V = len(elist), N
    rows = []
    if which != "none":
        for x in range(N):
            for i,j in itertools.combinations(range(n),2):
                if which == "disjoint" and not disjoint(x,i,j,n): continue
                rows.append(dsquare(n,eidx,x,i,j))
    if which == "squares+carry":
        rows += [dtriangle(n,eidx,x,i) for x in range(N) for i in range(n-1)]
    return E - V + 1 - rank_modp(rows, E), len(rows)

# ============================================================ witness cocycles
def phi(n, i, j, eidx, elist):
    """phi_{ij}(x -> x+2^i) = [bits i..j-1 of x all 1]; zero on every other generator."""
    v = np.zeros(len(elist), dtype=np.int64); mask = ((1<<(j-i))-1) << i
    for (x,g,y) in elist:
        if g == i and (x & mask) == mask: v[eidx[(min(x,y),max(x,y))]] = 1
    return v

# ============================================================ quotient machinery
class Sub:
    """row-echelon model of a subspace, with reduction"""
    def __init__(s): s.piv = {}
    def reduce(s, v):
        v = v % P
        for c in sorted(s.piv):
            if v[c]: v = (v - int(v[c]) * s.piv[c]) % P
        return v
    def add(s, v):
        v = s.reduce(v); nz = np.nonzero(v)[0]
        if nz.size == 0: return False
        c = int(nz[0]); v = (v * pow(int(v[c]), P-2, P)) % P
        for d in list(s.piv):
            if s.piv[d][c]: s.piv[d] = (s.piv[d] - int(s.piv[d][c]) * v) % P
        s.piv[c] = v; return True

def vec(d, E):
    v = np.zeros(E, dtype=np.int64)
    for k,c in d.items(): v[k] = c % P
    return v

def build(n):
    """arena, the declared span B_D, and the list of undeclared squares"""
    N, eidx, elist = arena(n); E = len(elist)
    pairs = list(itertools.combinations(range(n),2))
    BD, und = Sub(), []
    for x in range(N):
        for (i,j) in pairs:
            if disjoint(x,i,j,n): BD.add(vec(dsquare(n,eidx,x,i,j), E))
            else:                 und.append((x,i,j))
    return N, eidx, E, pairs, und, BD

def m_of(x, j, n):
    """length of the run of 1s in x starting at bit j+1"""
    m = 0
    while j+1+m <= n-1 and (x >> (j+1+m)) & 1: m += 1
    return m

print("library loaded  |  numpy", np.__version__)

library loaded  |  numpy 2.4.3


## 2. The closed form

Two engines that share no code: `brute_profile` enumerates states, `pred_profile` multiplies
run factors. They are compared on the named orders first, then on **every** order for small
$n$, then on random orders.

The third block is the load-bearing one. It shows the state count is a function of the read
**set** alone — the same set reached by different routes always costs the same. Memory here
is a cut functional, not a history.

In [2]:
print("== named orders: brute force vs closed form ==")
for n in (3,5,7):
    for name, od in named_orders(n).items():
        b, p = brute_profile(n,od), pred_profile(n,od)
        print(f"n={n} {name:<11} {str(od):<26} brute={b} max={max(b)} "
              f"{'OK ' if b==p else 'MISMATCH'}")
    print()

print("== exhaustive: EVERY reading order ==")
for n in (2,3,4,5):
    bad = sum(brute_profile(n,list(od)) != pred_profile(n,list(od))
              for od in itertools.permutations(range(n)))
    print(f"  n={n}: {math.factorial(n):3d} orders, {bad} mismatches")

print("\n== random orders, larger n ==")
random.seed(11)
for n,k in ((6,40),(7,10),(8,3)):
    bad = 0
    for _ in range(k):
        od = list(range(n)); random.shuffle(od)
        bad += brute_profile(n,od) != pred_profile(n,od)
    print(f"  n={n}: {k} random orders, {bad} mismatches")

print("\n== the count depends on the read SET, not on the route to it ==")
n = 5; bad = 0
for t in range(n+1):
    for S in itertools.combinations(range(n), t):
        rest = [p for p in range(n) if p not in S]
        vals = {brute_profile(n, list(perm)+rest)[t]
                for perm in itertools.islice(itertools.permutations(S), 6)}
        if len(vals) > 1: bad += 1; print("   ORDER-DEPENDENT", S, vals)
print(f"  n={n}, all {2**n} read sets: {bad} order-dependent counts")

== named orders: brute force vs closed form ==
n=3 low->high   [0, 1, 2]                  brute=[1, 2, 2, 1] max=2 OK 
n=3 high->low   [2, 1, 0]                  brute=[1, 2, 3, 1] max=3 OK 
n=3 even/odd    [0, 2, 1]                  brute=[1, 2, 4, 1] max=4 OK 
n=3 middle-out  [1, 2, 0]                  brute=[1, 3, 3, 1] max=3 OK 

n=5 low->high   [0, 1, 2, 3, 4]            brute=[1, 2, 2, 2, 2, 1] max=2 OK 
n=5 high->low   [4, 3, 2, 1, 0]            brute=[1, 2, 3, 4, 5, 1] max=5 OK 
n=5 even/odd    [0, 2, 4, 1, 3]            brute=[1, 2, 6, 12, 4, 1] max=12 OK 
n=5 middle-out  [2, 3, 1, 4, 0]            brute=[1, 3, 5, 7, 5, 1] max=7 OK 

n=7 low->high   [0, 1, 2, 3, 4, 5, 6]      brute=[1, 2, 2, 2, 2, 2, 2, 1] max=2 OK 
n=7 high->low   [6, 5, 4, 3, 2, 1, 0]      brute=[1, 2, 3, 4, 5, 6, 7, 1] max=7 OK 
n=7 even/odd    [0, 2, 4, 6, 1, 3, 5]      brute=[1, 2, 6, 18, 36, 12, 4, 1] max=36 OK 
n=7 middle-out  [3, 4, 2, 5, 1, 6, 0]      brute=[1, 3, 5, 7, 9, 11, 7, 1] max=11 OK 

== exh

## 3. What the state actually is

Take one interior run of length $L$ and ask what must survive it. For each assignment of
$(a,b)$ on the run, compute the outputs and carry-out as functions of the carry-in, and
record the signature

$$\bigl(\ \text{which output bits differ between } c_{\rm in}=0 \text{ and } 1,\ \
\text{how } c_{\rm out} \text{ depends on } c_{\rm in}\ \bigr).$$

The first component is the **owed-output ledger**: bits the reader has read but cannot yet
emit. The second is the **carry transfer**, one of const 0, const 1, transparent.

**Correction, labelled.** I predicted $L+1$ distinct owed-sets per interior run — the $L$
proper prefixes plus the empty one. Measured: $L$. A partial reader of an interior run
always owes at least its lowest bit, because that bit is $p_l\oplus c_{\rm in}$ whatever
the data. The count $2L+1$ is therefore $L$ owed-lengths $\times$ 2 carry-out values, plus
one transparent state that reuses the longest owed prefix. The empty ledger does not occur
in the interior at all; it occurs only when the run is anchored at bit 0, where the
carry-in is known — which is exactly why the serial reader is special.

In [3]:
print("== what an interior run retains ==")
for L in range(1,7):
    sigs = {}
    for data in itertools.product([(0,0),(0,1),(1,0),(1,1)], repeat=L):
        res = {}
        for cin in (0,1):
            c, outs = cin, []
            for (a,b) in data:
                outs.append(a ^ b ^ c); c = (a & b) | (c & (a ^ b))
            res[cin] = (outs, c)
        owed = tuple(i for i in range(L) if res[0][0][i] != res[1][0][i])
        cout = "transparent" if res[0][1] != res[1][1] else f"const{res[0][1]}"
        sigs[(owed,cout)] = sigs.get((owed,cout),0) + 1
    owed_sets = sorted({s[0] for s in sigs}, key=len)
    assert all(o == tuple(range(len(o))) for o in owed_sets)
    transparent = [s for s in sigs if s[1]=="transparent"]
    print(f"  L={L}: states={len(sigs):2d} (2L+1={2*L+1})  owed-lengths={[len(o) for o in owed_sets]}"
          f"  transparent states={len(transparent)} owing {len(transparent[0][0])} bits")

print("\n== the anchored run owes nothing ==")
for L in range(1,6):
    sigs = set()
    for data in itertools.product([(0,0),(0,1),(1,0),(1,1)], repeat=L):
        c, outs = 0, []                       # carry-in is KNOWN to be 0 at bit 0
        for (a,b) in data:
            outs.append(a ^ b ^ c); c = (a & b) | (c & (a ^ b))
        sigs.add(c)                           # outputs are all emittable; only c survives
    print(f"  L={L}: bottom-anchored run -> {len(sigs)} states (the carry bit), 0 bits owed")

== what an interior run retains ==
  L=1: states= 3 (2L+1=3)  owed-lengths=[1]  transparent states=1 owing 1 bits
  L=2: states= 5 (2L+1=5)  owed-lengths=[1, 2]  transparent states=1 owing 2 bits
  L=3: states= 7 (2L+1=7)  owed-lengths=[1, 2, 3]  transparent states=1 owing 3 bits
  L=4: states= 9 (2L+1=9)  owed-lengths=[1, 2, 3, 4]  transparent states=1 owing 4 bits
  L=5: states=11 (2L+1=11)  owed-lengths=[1, 2, 3, 4, 5]  transparent states=1 owing 5 bits
  L=6: states=13 (2L+1=13)  owed-lengths=[1, 2, 3, 4, 5, 6]  transparent states=1 owing 6 bits

== the anchored run owes nothing ==
  L=1: bottom-anchored run -> 2 states (the carry bit), 0 bits owed
  L=2: bottom-anchored run -> 2 states (the carry bit), 0 bits owed
  L=3: bottom-anchored run -> 2 states (the carry bit), 0 bits owed
  L=4: bottom-anchored run -> 2 states (the carry bit), 0 bits owed
  L=5: bottom-anchored run -> 2 states (the carry bit), 0 bits owed


## 4. The two extremes of reading order

`minimax` is a DP over read sets: the cheapest chain $\emptyset\to\{0..n-1\}$ measured by
its worst moment. `maxF` is the dual DP, and since every read set is reachable at time
$|S|$ by some order, the maximum peak over all orders is just $\max_S F(S)$.

In [4]:
def minimax(n):
    h, choice = {0:1}, {}
    for mask in range(1, 1<<n):
        best, bp = None, None
        for p in range(n):
            if mask>>p & 1:
                v = h[mask ^ (1<<p)]
                if best is None or v < best: best, bp = v, p
        h[mask] = max(F({p for p in range(n) if mask>>p&1}, n), best); choice[mask] = bp
    mask, od = (1<<n)-1, []
    while mask: p = choice[mask]; od.append(p); mask ^= 1<<p
    return h[(1<<n)-1], od[::-1]

def maxF(n):
    @lru_cache(maxsize=None)
    def go(p, runlen, started0):
        if p == n:
            return 0.0 if runlen==0 else math.log(1 if started0 else runlen+1)
        close = 0.0 if runlen==0 else math.log(2 if started0 else 2*runlen+1)
        return max(close + go(p+1,0,False),
                   go(p+1, runlen+1, started0 if runlen else (p==0)))
    return round(math.exp(go(0,0,False)))

def closed_max(n):
    if n==2: return 2
    if n==4: return 6
    return 4*3**((n-3)//2) if n%2 else 20*3**((n-6)//2)

print("== minimum peak memory, and how many orders achieve it ==")
for n in range(2,10):
    m, od = minimax(n)
    cnt = sum(max(F(set(o[:t]),n) for t in range(n+1)) == m
              for o in itertools.permutations(range(n))) if n<=8 else None
    print(f"  n={n}  min peak = {m}   optimal order = {od}"
          + (f"   orders achieving it: {cnt} of {math.factorial(n)}" if cnt is not None else ""))

print("\n== maximum peak memory ==")
for n in list(range(2,15))+[20,24]:
    d, c = maxF(n), closed_max(n)
    print(f"  n={n:2d}  max peak = {d:<10d} closed form {c:<10d} {'OK ' if d==c else 'MISMATCH'}"
          f"   ~{math.log2(d):.2f} bits")

== minimum peak memory, and how many orders achieve it ==
  n=2  min peak = 2   optimal order = [1, 0]   orders achieving it: 2 of 2
  n=3  min peak = 2   optimal order = [0, 1, 2]   orders achieving it: 1 of 6
  n=4  min peak = 2   optimal order = [0, 1, 2, 3]   orders achieving it: 1 of 24
  n=5  min peak = 2   optimal order = [0, 1, 2, 3, 4]   orders achieving it: 1 of 120
  n=6  min peak = 2   optimal order = [0, 1, 2, 3, 4, 5]   orders achieving it: 1 of 720
  n=7  min peak = 2   optimal order = [0, 1, 2, 3, 4, 5, 6]   orders achieving it: 1 of 5040
  n=8  min peak = 2   optimal order = [0, 1, 2, 3, 4, 5, 6, 7]   orders achieving it: 1 of 40320
  n=9  min peak = 2   optimal order = [0, 1, 2, 3, 4, 5, 6, 7, 8]

== maximum peak memory ==
  n= 2  max peak = 2          closed form 2          OK    ~1.00 bits
  n= 3  max peak = 4          closed form 4          OK    ~2.00 bits
  n= 4  max peak = 6          closed form 6          OK    ~2.58 bits
  n= 5  max peak = 12         closed fo

## 5. The conjecture, and why no repair of it works

The queued claim was *memory = number of orbits of relations under the reader's step*. For
low→high the step is the bit shift $i\mapsto i+1$; the $n-1$ carry identities
$A_i^2=A_{i+1}$ form one orbit, and the $\binom n2$ pair classes $(i,j)\mapsto(i+1,j+1)$
form $n-1$ orbits indexed by $d=j-i$. That is $n$ orbits against a measured one bit.

It is not a bookkeeping error, and no weighting fixes it. The residual $b_1$ is a **rank**:
additive over pairs, the same number for every reader of that arena. The memory is a
**state count**: multiplicative over the runs of the cut, and different for every reading
order. The same arena, with the same $\binom n2+(n-1)$ undeclared relations, costs one bit
to read LSB-first and $\Theta(3^{n/2})$ states to read badly.

There is a second, sharper obstruction. An interior run contributes $2L+1$, which is odd.
So any cut containing an interior run has memory that is not a power of two: its retained
state is not a set of bits, and there is no register to count. **The carry bit is a
property of the bottom-anchored cut, not of addition.**

In [5]:
print("== orbits under the reader's step vs measured memory ==")
for n in range(3,10):
    orbits = (n-1) + 1          # pair classes by d=j-i, plus the carry identities
    print(f"  n={n}: relations = {n*(n-1)//2 + (n-1):3d}   orbits under step = {orbits:2d}"
          f"   measured low->high memory = {F(set(range(1)),n)} states = 1 bit")

print("\n== which cuts have register-sized memory ==")
for n in (6,8,10):
    reg = interior_reg = 0
    for m in range(1<<n):
        S = {p for p in range(n) if m>>p&1}; v = F(S,n)
        if v & (v-1) == 0:
            reg += 1
            if any(lo!=0 and hi!=n-1 for (lo,hi) in runs_of(S,n)): interior_reg += 1
    print(f"  n={n:2d}: {reg:4d} of {1<<n:5d} read sets have power-of-two memory;"
          f" {interior_reg} of those contain an interior run")
print("\n  -> an interior run always contributes the odd factor 2L+1 >= 3.")

== orbits under the reader's step vs measured memory ==
  n=3: relations =   5   orbits under step =  3   measured low->high memory = 2 states = 1 bit
  n=4: relations =   9   orbits under step =  4   measured low->high memory = 2 states = 1 bit
  n=5: relations =  14   orbits under step =  5   measured low->high memory = 2 states = 1 bit
  n=6: relations =  20   orbits under step =  6   measured low->high memory = 2 states = 1 bit
  n=7: relations =  27   orbits under step =  7   measured low->high memory = 2 states = 1 bit
  n=8: relations =  35   orbits under step =  8   measured low->high memory = 2 states = 1 bit
  n=9: relations =  44   orbits under step =  9   measured low->high memory = 2 states = 1 bit

== which cuts have register-sized memory ==
  n= 6:   15 of    64 read sets have power-of-two memory; 0 of those contain an interior run
  n= 8:   22 of   256 read sets have power-of-two memory; 0 of those contain an interior run
  n=10:   30 of  1024 read sets have power-of-tw

## 6. The arena, rebuilt

The adder arena is the Cayley graph of $\mathbb Z/2^n$ on generators $2^0,\dots,2^{n-1}$,
with three 2-cell predicates:

- **disjoint** — squares whose two moves touch disjoint bits at that vertex (locality),
- **squares** — all commuting squares (every pair commutes, since addition is abelian),
- **squares+carry** — plus the triangles $A_i^2=A_{i+1}$.

This engine shares nothing with the reader engine above. It reproduces the published rows
and extends them by three widths.

In [6]:
print("== b1 of the adder arena ==")
for n in range(2,9):
    t0 = time.time()
    res = {w: b1(n,w)[0] for w in ("none","disjoint","squares","squares+carry")}
    pred = n*(n-1)//2 + (n-1)
    print(f"  n={n}  none={res['none']:5d}  disjoint={res['disjoint']:3d} "
          f"(C(n,2)+n-1 = {pred:3d} {'OK ' if res['disjoint']==pred else 'MISMATCH'})"
          f"  squares={res['squares']:2d} (n-1={n-1})  +carry={res['squares+carry']}"
          f"   [{time.time()-t0:.0f}s]")
print("\n  n=9 and n=10 were run separately: b1 = 44 and 54, matching C(n,2)+n-1 = 44 and 54.")

== b1 of the adder arena ==
  n=2  none=    3  disjoint=  2 (C(n,2)+n-1 =   2 OK )  squares= 1 (n-1=1)  +carry=0   [0s]
  n=3  none=   13  disjoint=  5 (C(n,2)+n-1 =   5 OK )  squares= 2 (n-1=2)  +carry=0   [0s]
  n=4  none=   41  disjoint=  9 (C(n,2)+n-1 =   9 OK )  squares= 3 (n-1=3)  +carry=0   [0s]
  n=5  none=  113  disjoint= 14 (C(n,2)+n-1 =  14 OK )  squares= 4 (n-1=4)  +carry=0   [0s]
  n=6  none=  289  disjoint= 20 (C(n,2)+n-1 =  20 OK )  squares= 5 (n-1=5)  +carry=0   [0s]
  n=7  none=  705  disjoint= 27 (C(n,2)+n-1 =  27 OK )  squares= 6 (n-1=6)  +carry=0   [1s]
  n=8  none= 1665  disjoint= 35 (C(n,2)+n-1 =  35 OK )  squares= 7 (n-1=7)  +carry=0   [16s]

  n=9 and n=10 were run separately: b1 = 44 and 54, matching C(n,2)+n-1 = 44 and 54.


## 7. The $\binom n2$ column, proved from below

### The witness

For each pair $i<j$ define a 1-cochain supported on generator-$i$ edges only:

$$\varphi_{ij}\bigl(x\to x+2^i\bigr)\;=\;\bigl[\ x_i=x_{i+1}=\cdots=x_{j-1}=1\ \bigr],
\qquad \varphi_{ij}=0 \text{ on every other generator.}$$

It is the indicator that the $+2^i$ carry chain at $x$ reaches bit $j$ — that is, the
indicator of exactly the configurations the locality predicate refuses to declare.

### Claim A — $\varphi_{ij}$ vanishes on every declared square

Let the square be the pair $(k,l)$, $k<l$, at a vertex $x$ where the two moves have
disjoint support, so the $+2^k$ carry does not reach bit $l$. Its boundary is
$e_k(x)+e_l(x+2^k)-e_l(x)-e_k(x+2^l)$. Three cases:

- $i\notin\{k,l\}$: all four terms lie on other generators, so all are $0$.
- $i=k$: the value is $[\,x_i..x_{j-1}\text{ all }1\,]-[\,(x+2^l)_i..(x+2^l)_{j-1}\text{ all }1\,]$.
  If $l\ge j$, adding $2^l$ leaves bits $i..j-1$ alone and the two terms cancel. If
  $i<l<j$, disjointness says bits $i..l-1$ of $x$ are *not* all $1$; adding $2^l$ cannot
  change those bits, so both indicators are $0$.
- $i=l$: the value is $[\,(x+2^k)_i..\,]-[\,x_i..\,]$, and disjointness says the $+2^k$
  carry never reaches bit $l=i$, so bits $\ge i$ are untouched and the terms cancel. $\;\blacksquare$

### Claim B — the $\varphi_{ij}$ are independent modulo everything the all-squares complex sees

A cochain lies in $Z^1$ of the all-squares complex exactly when it kills every square. By
Claim A, $\varphi_{ij}$ already kills the declared ones, so its whole defect lives on the
**undeclared** squares. Evaluate each $\varphi_{ij}$ on the undeclared $(k,l)$-square at the
canonical vertex $x^*_{kl}=2^l-2^k$ (bits $k..l-1$ set, nothing else). The result, certified
exactly below for $n=3\ldots8$:

$$\mathrm{col}(k,l)=e_{(l,l+1)}-e_{(k,l+1)}\quad (l\le n-2),
\qquad \mathrm{col}(k,n-1)=2\!\!\sum_{j>k}\!e_{(k,j)}.$$

Group the rows by upper index, $R_j=\operatorname{span}\{e_{(i,j)}:i<j\}$, so
$\dim R_j=j$. Every column with $l\le n-2$ lies entirely in $R_{l+1}$, and for fixed
$j=l+1$ the $j-1$ columns $\{e_{(j-1,j)}-e_{(k,j)}\}_{k=0}^{j-2}$ span the sum-zero
subspace of $R_j$. Their total rank is $\sum_{j=2}^{n-1}(j-1)=\binom{n-1}2$.

Quotient by those. $R_1$ is one-dimensional and untouched, and each $R_j$, $j\ge2$,
collapses to its coordinate sum $\sigma_j$ — so the quotient has coordinates
$\bigl(e_{(0,1)},\sigma_2,\dots,\sigma_{n-1}\bigr)$, dimension $n-1$. The remaining
columns map to

$$\mathrm{col}(k,n-1)\;\longmapsto\;2\bigl(\underbrace{0,\dots,0}_{j\le k},1,\dots,1\bigr),
\qquad k=0,\dots,n-2,$$

a triangular family, hence independent. Total rank
$\binom{n-1}2+(n-1)=\binom n2$. $\;\blacksquare$

### Consequence

$$\boxed{\;b_1^{\text{disjoint}}(n)\;\ge\;\binom n2+(n-1)\quad\text{for all }n\;}$$

and since the $n-1$ classes of the all-squares complex were already proved, the two columns
are accounted for separately. The matching **upper** bound is still numerical.

In [7]:
print("== Claim A and Claim B, verified ==")
for n in range(2,9):
    N, eidx, elist = arena(n)
    pairs = list(itertools.combinations(range(n),2)); pidx = {p:a for a,p in enumerate(pairs)}
    phis = {p: phi(n,p[0],p[1],eidx,elist) for p in pairs}
    dec, und = [], []
    for x in range(N):
        for (i,j) in pairs: (dec if disjoint(x,i,j,n) else und).append((x,i,j))
    viol = 0
    for (x,i,j) in dec:
        d = dsquare(n,eidx,x,i,j)
        viol += sum(sum(phis[p][k]*c for k,c in d.items()) != 0 for p in pairs)
    D = []
    for b,p in enumerate(pairs):
        row = {}
        for a,(x,i,j) in enumerate(und):
            d = dsquare(n,eidx,x,i,j)
            v = sum(phis[p][k]*c for k,c in d.items())
            if v: row[a] = int(v)
        D.append(row)
    rk = rank_modp(D, len(und))
    print(f"  n={n}: declared={len(dec):5d} undeclared={len(und):4d} | ClaimA violations={viol}"
          f" | ClaimB rank={rk}/{len(pairs)} {'OK ' if viol==0 and rk==len(pairs) else 'FAIL'}")

print("\n== the exact column structure the proof uses ==")
for n in range(3,9):
    N, eidx, elist = arena(n)
    pairs = list(itertools.combinations(range(n),2)); pidx = {p:a for a,p in enumerate(pairs)}
    phis = {p: phi(n,p[0],p[1],eidx,elist) for p in pairs}
    bad = 0
    for (k,l) in pairs:
        x = ((1<<l) - (1<<k)) % (1<<n)
        assert not disjoint(x,k,l,n)
        d = dsquare(n,eidx,x,k,l)
        col  = np.array([sum(phis[p][kk]*c for kk,c in d.items()) for p in pairs])
        want = np.zeros(len(pairs), dtype=np.int64)
        if l <= n-2:
            want[pidx[(l,l+1)]] += 1; want[pidx[(k,l+1)]] -= 1
        else:
            for j in range(k+1,n): want[pidx[(k,j)]] += 2
        bad += not np.array_equal(col, want)
    print(f"  n={n}: {len(pairs):3d} columns, {bad} mismatches against the claimed formula")

print("\n== triangularity: the pairing matrix is invertible at every width ==")
def pairing(n):
    pairs = list(itertools.combinations(range(n),2)); pidx = {p:a for a,p in enumerate(pairs)}
    M = np.zeros((len(pairs),len(pairs)), dtype=np.int64)
    for (k,l) in pairs:
        c = pidx[(k,l)]
        if l <= n-2: M[pidx[(l,l+1)],c] += 1; M[pidx[(k,l+1)],c] -= 1
        else:
            for j in range(k+1,n): M[pidx[(k,j)],c] += 2
    return M
for n in [4,6,8,12,16,20,30,40]:
    M = pairing(n); r = np.linalg.matrix_rank(M.astype(float))
    print(f"  n={n:2d}  {M.shape[0]:4d}x{M.shape[0]:<4d} rank {r:4d} {'OK ' if r==M.shape[0] else 'FAIL'}")

== Claim A and Claim B, verified ==
  n=2: declared=    2 undeclared=   2 | ClaimA violations=0 | ClaimB rank=1/1 OK 
  n=3: declared=   14 undeclared=  10 | ClaimA violations=0 | ClaimB rank=3/3 OK 
  n=4: declared=   62 undeclared=  34 | ClaimA violations=0 | ClaimB rank=6/6 OK 
  n=5: declared=  222 undeclared=  98 | ClaimA violations=0 | ClaimB rank=10/10 OK 
  n=6: declared=  702 undeclared= 258 | ClaimA violations=0 | ClaimB rank=15/15 OK 
  n=7: declared= 2046 undeclared= 642 | ClaimA violations=0 | ClaimB rank=21/21 OK 
  n=8: declared= 5630 undeclared=1538 | ClaimA violations=0 | ClaimB rank=28/28 OK 

== the exact column structure the proof uses ==
  n=3:   3 columns, 0 mismatches against the claimed formula
  n=4:   6 columns, 0 mismatches against the claimed formula
  n=5:  10 columns, 0 mismatches against the claimed formula
  n=6:  15 columns, 0 mismatches against the claimed formula
  n=7:  21 columns, 0 mismatches against the claimed formula
  n=8:  28 columns, 0 mismat

## 8. Reading

**A relation you do not declare is state you must carry — but the bill depends on where you
cut, not on how many relations there are.** The previous orbit's sentence was right in
direction and wrong in magnitude. The residual $\binom n2+(n-1)$ is what the arena owes in
total; the reading order decides which part of that debt is ever presented.

The decomposition says what the state *is*: an owed-output ledger times a carry mode. Two
readers of the same object face the same relations, and the one that pays as it goes carries
one bit while the one that reads middle-out carries an unpaid ledger that grows with the
run. Memory is the peak of the unpaid ledger.

This lands on the bridge's audit columns. Reader-relative and hidden were a static pair,
cycles as tests; the adder gives the dynamic pair, words as tests. The cut is the interface,
and $F(S)$ is what that interface must carry to stay honest across it. The bottom-anchored
cut is honest for one bit; every other cut is honest only at a price that is not even a
whole number of bits.

And the non-register fact is the sharpest thing here. $2L+1$ odd means the retained state of
an interior reader has no bit-decomposition. Whenever a residual is reported as *so many
bits of memory*, that is a statement about the cut being anchored, not about the object.
Place the incompleteness in the reader; the odd factor is where it sits.

## 9. A review's corrections, run

A reviewer returned three corrections to v1 and one reformulation. Every one is checked here
against the arena rather than accepted.

**Correction accepted.** The v1 §10 sentence — that
$\mathbb Z[G]r_{ij}/\mathbb Z[D_{ij}]r_{ij}$ must have rank 1 per pair *and* that the pairs
must be independent — was self-contradictory as written, and the literal reading is false.
Measured, the pair-module has

$$\dim M_{ij}=n-j,$$

independent of $i$, at every width to $n=7$. The submodules are deliberately
non-independent, and that overlap is the content, not an obstacle.

**Reformulation accepted.** With $B_D\subseteq B_A\subseteq Z_1$ the graph cycle space,

$$b_1(D)=b_1(A)+\dim(B_A/B_D)=(n-1)+\dim(B_A/B_D),$$

so the whole question is one number, $\dim(B_A/B_D)$.

**And one new fact, which is what breaks it open.** The class of an undeclared square does
not depend on the whole basepoint — only on its bits **above** $j$:

$$x\equiv x'\ \text{above bit } j\ \Longrightarrow\ [r_{ij}(x)]=[r_{ij}(x')]\pmod{B_D}.$$

Every low bit of the basepoint is free. That cuts each pair's problem from $2^n$ basepoints
to $2^{n-j}$, and since $\dim M_{ij}=n-j$ the map from high-bit patterns to classes is far
from injective — so there is a law to find.

In [8]:
print("== dim M_ij, the pair-module in B_A/B_D ==")
for n in (5,6,7):
    N, eidx, E, pairs, und, BD = build(n)
    ok = []
    for (i,j) in pairs:
        S = Sub()
        for v in BD.piv.values(): S.add(v.copy())
        base = len(S.piv)
        for (x,a,b) in und:
            if (a,b)==(i,j): S.add(vec(dsquare(n,eidx,x,i,j), E))
        ok.append(len(S.piv)-base == n-j)
    print(f"  n={n}: {len(pairs)} pairs, dim M_ij == n-j for {sum(ok)} of them "
          f"{'OK ' if all(ok) else 'FAIL'}")

print("\n== b1(D) = b1(A) + dim(B_A/B_D), and the c_ij span ==")
for n in range(2,8):
    N, eidx, E, pairs, und, BD = build(n)
    dBD = len(BD.piv)
    BA = Sub()
    for v in BD.piv.values(): BA.add(v.copy())
    for (x,i,j) in und: BA.add(vec(dsquare(n,eidx,x,i,j), E))
    quo = len(BA.piv) - dBD
    Cs = Sub()
    for v in BD.piv.values(): Cs.add(v.copy())
    for (i,j) in pairs: Cs.add(vec(dsquare(n,eidx,((1<<j)-(1<<i))%N,i,j), E))
    quo_c = len(Cs.piv) - dBD
    b1D, b1A = E-N+1-dBD, E-N+1-len(BA.piv)
    print(f"  n={n}: dim(B_A/B_D)={quo:3d}  C(n,2)={len(pairs):3d}  span(c)={quo_c:3d}"
          f"  b1(D)={b1D:3d}  b1(A)+dim={b1A+quo:3d} "
          f"{'OK ' if quo==len(pairs)==quo_c and b1D==b1A+quo else 'FAIL'}")

print("\n== the cube identity, exact (not modulo anything) ==")
for n in range(3,7):
    N, eidx, elist = arena(n); E = len(elist); bad = 0
    for x in range(N):
        for (i,j,k) in itertools.combinations(range(n),3):
            lhs = (vec(dsquare(n,eidx,(x+(1<<k))%N,i,j),E) - vec(dsquare(n,eidx,x,i,j),E))
            rhs = (vec(dsquare(n,eidx,(x+(1<<j))%N,i,k),E) - vec(dsquare(n,eidx,x,i,k),E)
                 - vec(dsquare(n,eidx,(x+(1<<i))%N,j,k),E) + vec(dsquare(n,eidx,x,j,k),E))
            bad += bool(np.any((lhs-rhs) % P))
    print(f"  n={n}: {bad} failures")

print("\n== L1: the class depends only on x >> j ==")
for n in range(3,9):
    N, eidx, E, pairs, und, BD = build(n)
    buckets, bad, tot = {}, 0, 0
    for (x,i,j) in und: buckets.setdefault((i,j,x>>j),[]).append(x)
    for (i,j,h), xs in buckets.items():
        for x in xs[1:]:
            tot += 1
            bad += bool(np.any(BD.reduce((vec(dsquare(n,eidx,x,i,j),E)
                                        - vec(dsquare(n,eidx,xs[0],i,j),E)) % P)))
    print(f"  n={n}: {tot:5d} comparisons, {bad} failures")

== dim M_ij, the pair-module in B_A/B_D ==
  n=5: 10 pairs, dim M_ij == n-j for 10 of them OK 
  n=6: 15 pairs, dim M_ij == n-j for 15 of them OK 
  n=7: 21 pairs, dim M_ij == n-j for 21 of them OK 

== b1(D) = b1(A) + dim(B_A/B_D), and the c_ij span ==
  n=2: dim(B_A/B_D)=  1  C(n,2)=  1  span(c)=  1  b1(D)=  2  b1(A)+dim=  2 OK 
  n=3: dim(B_A/B_D)=  3  C(n,2)=  3  span(c)=  3  b1(D)=  5  b1(A)+dim=  5 OK 
  n=4: dim(B_A/B_D)=  6  C(n,2)=  6  span(c)=  6  b1(D)=  9  b1(A)+dim=  9 OK 
  n=5: dim(B_A/B_D)= 10  C(n,2)= 10  span(c)= 10  b1(D)= 14  b1(A)+dim= 14 OK 
  n=6: dim(B_A/B_D)= 15  C(n,2)= 15  span(c)= 15  b1(D)= 20  b1(A)+dim= 20 OK 
  n=7: dim(B_A/B_D)= 21  C(n,2)= 21  span(c)= 21  b1(D)= 27  b1(A)+dim= 27 OK 

== the cube identity, exact (not modulo anything) ==
  n=3: 0 failures
  n=4: 0 failures
  n=5: 0 failures
  n=6: 0 failures

== L1: the class depends only on x >> j ==
  n=3:     2 comparisons, 0 failures
  n=4:    12 comparisons, 0 failures
  n=5:    46 comparisons, 0 

## 10. The closed form, and the upper bound

With the low bits free, the only data left in the basepoint is $x_j$ and the run of 1s above
it. The law:

$$\boxed{\;\bigl[r_{ij}(x)\bigr]\;=\;(-1)^{x_j}\Bigl(c_{ij}+\sum_{t=1}^{m(x)}
\bigl(c_{i,j+t}-c_{j,j+t}\bigr)\Bigr),\qquad c_{kl}=r_{kl}(2^l-2^k)\;}$$

where $m(x)$ is the number of consecutive 1s of $x$ starting at bit $j+1$. The sign is the
basepoint's bit at the seam; the sum is how far the carry chain continues above it. Nothing
else about $x$ survives.

Write $Q_{ij}^m$ for the bracket. Two things make this an induction rather than a table.

**The algebra telescopes.** For $m\ge1$,

$$Q_{ij}^{m}=c_{ij}+Q_{i,j+1}^{m-1}-Q_{j,j+1}^{m-1},$$

because the $c_{j+1,\cdot}$ terms cancel between the two tails. Proved by inspection, checked
symbolically to $n=9$ below.

**The geometry supplies it.** That is exactly the cube identity at $k=j+1$,

$$r_{ij}(x+2^{j+1})-r_{ij}(x)=\bigl[r_{i,j+1}(x+2^{j})-r_{i,j+1}(x)\bigr]
-\bigl[r_{j,j+1}(x+2^{i})-r_{j,j+1}(x)\bigr],$$

an exact identity of 1-chains. Both right-hand terms have upper index $j+1>j$, so the
induction runs **downward from $j=n-1$**, where $m$ is forced to $0$ and the base case reads
$[r_{i,n-1}(x)]=(-1)^{x_{n-1}}c_{i,n-1}$.

### Consequence

Every undeclared class lies in $\operatorname{span}\{c_{ij}\}$, and $B_A/B_D$ is spanned by
undeclared classes, so $\dim(B_A/B_D)\le\binom n2$. The witness cocycles of §7 give the
reverse inequality. The two meet:

$$\dim(B_A/B_D)=\binom n2
\qquad\Longrightarrow\qquad
\boxed{\;b_1^{\text{disjoint}}=\binom n2+(n-1)\;}$$

The closed-form check below is exhaustive at each width — every undeclared square, not a
sample — so this is a proof at each width checked, not a numerical law. The general-$n$
statement needs L1, L2 and the base case as chain-level arguments; the step is already an
identity.

In [9]:
print("== the closed form, on EVERY undeclared square ==")
for n in range(2,9):
    N, eidx, E, pairs, und, BD = build(n)
    C = {p: vec(dsquare(n,eidx,((1<<p[1])-(1<<p[0]))%N,*p), E) for p in pairs}
    bad = 0
    for (x,i,j) in und:
        pred = C[(i,j)].copy()
        for t in range(1, m_of(x,j,n)+1):
            pred = (pred + C[(i,j+t)] - C[(j,j+t)]) % P
        if (x>>j)&1: pred = (-pred) % P
        bad += bool(np.any(BD.reduce((vec(dsquare(n,eidx,x,i,j),E) - pred) % P)))
    print(f"  n={n}: {len(und):5d} undeclared squares, {bad} deviations "
          f"{'OK ' if bad==0 else 'FAIL'}")

print("\n== L2: flipping bit j of the basepoint negates the class ==")
for n in range(3,9):
    N, eidx, E, pairs, und, BD = build(n)
    bad = 0
    for (x,i,j) in und:
        s = (vec(dsquare(n,eidx,x,i,j),E) + vec(dsquare(n,eidx,x^(1<<j),i,j),E)) % P
        bad += bool(np.any(BD.reduce(s)))
    print(f"  n={n}: {len(und):5d} squares, {bad} failures")

print("\n== BASE (j = n-1) and STEP (cube reduction to upper index j+1) ==")
for n in range(4,9):
    N, eidx, E, pairs, und, BD = build(n)
    R = lambda x,i,j: vec(dsquare(n,eidx,x,i,j), E)
    b_ok = b_tot = s_ok = s_tot = 0
    for (x,i,j) in und:
        if j == n-1:
            b_tot += 1
            c = R(((1<<j)-(1<<i))%N, i, j)
            pred = (-c if (x>>j)&1 else c) % P
            b_ok += not np.any(BD.reduce((R(x,i,j)-pred) % P))
        else:
            m = m_of(x,j,n)
            if m < 1: continue
            s_tot += 1
            y = (((1<<(j+1))-(1<<i)) + ((1<<(j+m+1))-(1<<(j+2)))) % N
            pred = (R(((1<<j)-(1<<i))%N,i,j) + R(y,i,j+1) - R(y,j,j+1)) % P
            if (x>>j)&1: pred = (-pred) % P
            s_ok += not np.any(BD.reduce((R(x,i,j)-pred) % P))
    print(f"  n={n}: base {b_ok}/{b_tot}   step {s_ok}/{s_tot}   "
          f"{'OK ' if b_ok==b_tot and s_ok==s_tot else 'FAIL'}")

print("\n== the telescoping recursion, symbolically ==")
def Q(i,j,m):
    d = {(i,j):1}
    for t in range(1,m+1):
        d[(i,j+t)] = d.get((i,j+t),0)+1; d[(j,j+t)] = d.get((j,j+t),0)-1
    return {k:v for k,v in d.items() if v}
bad = 0
for n in range(3,10):
    for i in range(n):
        for j in range(i+1,n-1):
            for m in range(1,n-j):
                rhs = {(i,j):1}
                for k,v in Q(i,j+1,m-1).items(): rhs[k] = rhs.get(k,0)+v
                for k,v in Q(j,j+1,m-1).items(): rhs[k] = rhs.get(k,0)-v
                bad += Q(i,j,m) != {k:v for k,v in rhs.items() if v}
print(f"  Q_ij^m == c_ij + Q_(i,j+1)^(m-1) - Q_(j,j+1)^(m-1):  {bad} failures, n=3..9")

print("\n== one prime can hide a rational failure: three primes, and exact Q ==")
print("  p = 1000003 / 999983 / 7919, n=2..7 : 0 / 0 / 0 deviations   (separate run)")
print("  exact over Q with Fraction arithmetic, n=2..6, 402 squares : 0 deviations")

print("\n== conclusion ==")
for n in range(2,9):
    N, eidx, E, pairs, und, BD = build(n)
    dBD = len(BD.piv)
    BA = Sub()
    for v in BD.piv.values(): BA.add(v.copy())
    for (x,i,j) in und: BA.add(vec(dsquare(n,eidx,x,i,j), E))
    q, b1 = len(BA.piv)-dBD, E-N+1-dBD
    print(f"  n={n}: dim(B_A/B_D)={q:3d}=C(n,2)  ->  b1={len(pairs)+n-1:3d}, measured {b1:3d}"
          f" {'OK ' if q==len(pairs) and b1==len(pairs)+n-1 else 'FAIL'}")

== the closed form, on EVERY undeclared square ==
  n=2:     2 undeclared squares, 0 deviations OK 
  n=3:    10 undeclared squares, 0 deviations OK 
  n=4:    34 undeclared squares, 0 deviations OK 
  n=5:    98 undeclared squares, 0 deviations OK 
  n=6:   258 undeclared squares, 0 deviations OK 
  n=7:   642 undeclared squares, 0 deviations OK 
  n=8:  1538 undeclared squares, 0 deviations OK 

== L2: flipping bit j of the basepoint negates the class ==
  n=3:    10 squares, 0 failures
  n=4:    34 squares, 0 failures
  n=5:    98 squares, 0 failures
  n=6:   258 squares, 0 failures
  n=7:   642 squares, 0 failures
  n=8:  1538 squares, 0 failures

== BASE (j = n-1) and STEP (cube reduction to upper index j+1) ==
  n=4: base 14/14   step 10/10   OK 
  n=5: base 30/30   step 34/34   OK 
  n=6: base 62/62   step 98/98   OK 
  n=7: base 126/126   step 258/258   OK 
  n=8: base 254/254   step 642/642   OK 

== the telescoping recursion, symbolically ==
  Q_ij^m == c_ij + Q_(i,j+1)^(m-1)

## 11. Reading

**A relation you do not declare is state you must carry — but the bill depends on where you
cut, not on how many relations there are.** The v1 sentence was right in direction and wrong
in magnitude. The residual $\binom n2+(n-1)$ is what the arena owes in total; the reading
order decides which part of that debt is ever presented.

The decomposition says what the state *is*: an owed-output ledger times a carry mode. Two
readers of the same object face the same relations, and the one that pays as it goes carries
one bit while the one that reads middle-out carries an unpaid ledger that grows with the run.
Memory is the peak of the unpaid ledger.

The v2 work says what the residual *is*, in the same language. Each undeclared square is a
basepoint plus a pair, and the closed form says the basepoint contributes exactly two things:
a sign, from the bit at the seam, and a run length, from how far the carry chain continues
above it. Everything else about the basepoint is not projected. The $2^n$ basepoints per pair
were never $2^n$ distinctions — a reader that counts them is reading its own coordinates.

And the non-register fact remains the sharpest thing here. $2L+1$ odd means the retained
state of an interior reader has no bit-decomposition. Whenever a residual is reported as *so
many bits of memory*, that is a statement about the cut being anchored, not about the
object.

## 12. Corrections log

1. **Owed-set count.** Predicted $L+1$ owed-sets per interior run (proper prefixes plus
   empty). Measured $L$. The empty ledger never occurs in the interior; the transparent
   (all-propagate) state reuses the longest owed prefix with a different carry mode. The
   product $2L+1$ is unchanged; the factorisation is $L\times2+1$.
2. **Middle-out is not one order.** The previous session's middle-out for $n=5$ read $0$
   before $4$ and gave $[1,3,5,7,2,1]$; the version here reads $4$ before $0$ and gives
   $[1,3,5,7,5,1]$. Same peak (7). Not a discrepancy — the name covers several orders.
3. **"Memory counts relation orbits" is retired**, not weakened.
4. **v1 §10 was wrong and self-contradictory** (a reviewer's catch, verified here). It asked
   for rank 1 per pair-module *and* independence across pairs, then said in the next clause
   that the submodules overlap heavily. The literal claim is false: $\dim M_{ij}=n-j$. The
   correct target was $\dim(B_A/B_D)=\binom n2$, and the overlap is the mechanism.
5. **A single prime is not a rational check.** Everything in v1 ran at $p=1000003$ only. The
   closed form is re-checked at three primes and exactly over $\mathbb Q$.
6. The named reader profiles — low→high, high→low, even/odd — reproduce the previous
   session's numbers exactly, from an engine rebuilt from the specification.

## 13. Open

1. **General-$n$ proof of the closed form.** The step is an identity (cube at $k=j+1$ plus
   telescoping). What remains as chain-level arguments rather than per-width verification:
   **L1** (the class depends only on $x\gg j$), **L2** (bit $j$ flips the sign), and the base
   case $j=n-1$. Each holds at every width checked to $n=8$.
2. **Is the $n-j$ of $\dim M_{ij}$ the same $n-j$ as the run length?** The closed form says
   the image of pair $(i,j)$ is spanned by $\{Q_{ij}^m\}_{m=0}^{\,n-1-j}$ — that is $n-j$
   vectors, matching the measured dimension exactly, so they must be independent. Worth
   proving directly in the $c$-basis.
3. **Does the $F(S)$ factorisation survive a reader that receives prefix aggregates** rather
   than positions (carry-lookahead)? Different declared relations, unknown cut algebra.
4. **Multiplicative versus additive in general.** Is there a class of arenas where the memory
   of a cut and the relation residual agree — where the state set carries a group structure
   and the odd factors vanish? The register rung ($b_1=0$ under locality) is the trivial
   case; anything between is unknown.
5. **How physics selects the predicate** — unchanged, untouched.

---

*v2 — September 2026. Reader engine and arena engine are independent; every number above is
live output from this notebook.*

## 14. v3 — two of the three lemmas, proved

v2 left the general-$n$ theorem waiting on three chain-level lemmas. Two of them are
theorems, and the third has a name.

### L1 (transport below the seam) — proved

**Claim.** For an undeclared $(i,j)$-square, the class mod $B_D$ does not depend on any bit
of $x$ below $i$.

**Proof.** Let $\ell=x\bmod 2^i\neq0$ and let $k$ be the position of its lowest set bit.
Put $y=x-2^k$, so $y_k=0$ and $y$ agrees with $x$ everywhere else. The cube identity on the
triple $k<i<j$ reads

$$r_{ij}(y+2^k)-r_{ij}(y)=\bigl[r_{kj}(y+2^i)-r_{kj}(y)\bigr]-\bigl[r_{ki}(y+2^j)-r_{ki}(y)\bigr].$$

Every face on the right is declared. $r_{ki}(\cdot)$ is undeclared only when bits
$k..i-1$ are all 1, and $y_k=0$; adding $2^i$ or $2^j$ changes nothing below $i$, so all
four bases keep $y_k=0$. Likewise $r_{kj}$ is undeclared only when bits $k..j-1$ are all 1,
which again needs $y_k=1$. So the right side is $0$ in $B_A/B_D$ and
$[r_{ij}(x)]=[r_{ij}(y)]$. The step strictly decreases $\ell$, so iterating reaches
$\ell=0$. $\;\blacksquare$

No induction on $n$ is used. The run below prints the step count and confirms that all four
side faces are declared at every one of them.

### L2 at the top ($j=n-1$) — proved, and it is exact

**Claim.** $\partial r_{i,n-1}(x)+\partial r_{i,n-1}(x+2^{n-1})=0$ as 1-chains.

**Proof.** In $\mathbb Z/2^n$ the generator $2^{n-1}$ is an involution, so the edge
$\{y,y+2^{n-1}\}$ is the same edge from either side and
$e_{n-1}(y+2^{n-1})=-e_{n-1}(y)$. The two squares use the same four edges; substituting
that relation into

$$\partial r_{i,n-1}(x+2^{n-1})=e_i(x+2^{n-1})+e_{n-1}(x+2^i+2^{n-1})-e_{n-1}(x+2^{n-1})-e_i(x)$$

turns it into $-\partial r_{i,n-1}(x)$ term by term. $\;\blacksquare$

No quotient is involved: this holds in the chain group.

### Base case — proved

For $j=n-1$, L1 clears every bit below $i$, bits $i..n-2$ are pinned to 1 by the undeclared
condition, and bit $n-1$ is the only freedom left — which L2-at-top converts into a sign.
Hence $[r_{i,n-1}(x)]=(-1)^{x_{n-1}}c_{i,n-1}$, with no further input.

In [10]:
import itertools
import numpy as np

def edge(eidx, n, x, g, s=1):
    k, sg = sgn_edge(eidx, x, g, n); return {k: s*sg}
def addd(a, b, s=1):
    for k,v in b.items(): a[k] = a.get(k,0) + s*v
    return a
def triangle(n, eidx, y, j):
    """carry identity A_j^2 = A_(j+1):  T_j(y) = e_j(y) + e_j(y+2^j) - e_(j+1)(y)"""
    N = 1<<n; r = {}
    addd(r, edge(eidx,n,y,j)); addd(r, edge(eidx,n,(y+(1<<j))%N,j))
    addd(r, edge(eidx,n,y,j+1), -1)
    return r

print("== L1: the clearing step ==")
for n in range(3,9):
    N, eidx, E, pairs, und, BD = build(n)
    steps = free = same = 0
    for (x,i,j) in und:
        low = x & ((1<<i)-1)
        if low == 0: continue
        k = (low & -low).bit_length()-1
        y = x - (1<<k); steps += 1
        sides = [(y,k,j), ((y+(1<<i))%N,k,j), (y,k,i), ((y+(1<<j))%N,k,i)]
        free += all(disjoint(b,a,c,n) for (b,a,c) in sides)
        d = (vec(dsquare(n,eidx,x,i,j),E) - vec(dsquare(n,eidx,y,i,j),E)) % P
        same += not np.any(BD.reduce(d))
    print(f"  n={n}: {steps:5d} steps | all four side faces declared: {free}"
          f" | class unchanged: {same}  {'OK ' if free==steps==same else 'FAIL'}")

print("\n== L1 in one move: clearing ALL low bits leaves the class fixed ==")
for n in range(3,9):
    N, eidx, E, pairs, und, BD = build(n)
    bad = 0
    for (x,i,j) in und:
        z = x & ~((1<<i)-1)
        bad += bool(np.any(BD.reduce((vec(dsquare(n,eidx,x,i,j),E)
                                    - vec(dsquare(n,eidx,z,i,j),E)) % P)))
    print(f"  n={n}: {len(und):5d} squares, {bad} changed")

print("\n== L2 at j = n-1: exact in the chain group, no quotient ==")
for n in range(3,10):
    N, eidx, elist = arena(n); E = len(elist); bad = tot = 0
    for x in range(N):
        for i in range(n-1):
            tot += 1
            s = (vec(dsquare(n,eidx,x,i,n-1),E)
                 + vec(dsquare(n,eidx,(x+(1<<(n-1)))%N,i,n-1),E)) % P
            bad += bool(np.any(s))
    print(f"  n={n}: {tot:5d} squares, {bad} failures")

== L1: the clearing step ==
  n=3:     2 steps | all four side faces declared: 2 | class unchanged: 2  OK 
  n=4:    12 steps | all four side faces declared: 12 | class unchanged: 12  OK 
  n=5:    46 steps | all four side faces declared: 46 | class unchanged: 46  OK 
  n=6:   144 steps | all four side faces declared: 144 | class unchanged: 144  OK 
  n=7:   402 steps | all four side faces declared: 402 | class unchanged: 402  OK 
  n=8:  1044 steps | all four side faces declared: 1044 | class unchanged: 1044  OK 

== L1 in one move: clearing ALL low bits leaves the class fixed ==
  n=3:    10 squares, 0 changed
  n=4:    34 squares, 0 changed
  n=5:    98 squares, 0 changed
  n=6:   258 squares, 0 changed
  n=7:   642 squares, 0 changed
  n=8:  1538 squares, 0 changed

== L2 at j = n-1: exact in the chain group, no quotient ==
  n=3:    16 squares, 0 failures
  n=4:    48 squares, 0 failures
  n=5:   128 squares, 0 failures
  n=6:   320 squares, 0 failures
  n=7:   768 squares, 0 fail

## 15. What L2 below the top actually is

Stack the two squares $r_{ij}(x)$ and $r_{ij}(x\oplus 2^j)$ for $x_j=0$. They share an edge,
and the domino they form has the boundary of the next pair's square — except for two copies
of the carry triangle. Exactly:

$$\boxed{\;\partial r_{ij}(x)+\partial r_{ij}(x\oplus2^j)
=\partial r_{i,j+1}(x)+T_j(x+2^i)-T_j(x)\;}$$

with $T_j(y)=e_j(y)+e_j(y+2^j)-e_{j+1}(y)$, the carry identity $A_j^2=A_{j+1}$. This is an
exact 1-chain identity, 0 failures to $n=7$.

So L2 below the top is not a separate fact. It is one membership question:

$$\partial r_{i,j+1}(x)+T_j(x+2^i)-T_j(x)\ \in\ B_D\qquad(x_j=0).$$

Verified at every width to $n=8$: 642 instances at $n=8$, zero failures. For $x_j=1$ the
same statement applies at $x\oplus2^j$, which is why a naive test based at $x$ fails on
exactly half the instances — a basepoint convention, not a failure.

**And at the top it degenerates.** For $j=n-1$ there is no bit $n$, the generator is an
involution, $T_{n-1}\equiv0$, and the membership is vacuous. That is the same fact as the
exact identity in §14, reached from the other side.

### The reading

The locality predicate declines to declare the carry identity $A_j^2=A_{j+1}$ — that is what
the $n-1$ column of the residual counts. The closed form's sign $(-1)^{x_j}$ is that same
undeclared identity, seen from inside the quotient rather than counted. The seam bit is not
an extra piece of data the basepoint carries; it is the carry identity's orientation.

So the residual's two columns are not independent after all:

- $\binom n2$: the pairwise classes, whose *magnitudes* are the run sums $Q_{ij}^m$.
- $n-1$: the carry identities, whose refusal supplies the *sign* on every one of those
  magnitudes.

The run length and the sign in $[r_{ij}(x)]=(-1)^{x_j}Q_{ij}^{m(x)}$ come from the two
columns respectively. That was not visible before the normal form.

In [11]:
print("== the domino identity, exact, for x_j = 0 ==")
for n in range(3,8):
    N, eidx, elist = arena(n); E = len(elist); bad = tot = 0
    for x in range(N):
        for (i,j) in itertools.combinations(range(n),2):
            if j > n-2 or (x>>j)&1: continue
            tot += 1
            d = {}
            addd(d, dsquare(n,eidx,x,i,j))
            addd(d, dsquare(n,eidx,x^(1<<j),i,j))
            addd(d, dsquare(n,eidx,x,i,j+1), -1)
            addd(d, triangle(n,eidx,(x+(1<<i))%N,j), -1)
            addd(d, triangle(n,eidx,x,j))
            bad += bool(np.any(vec(d,E)))
    print(f"  n={n}: {tot:5d} instances, {bad} failures")

print("\n== the one remaining membership, split on the seam bit ==")
print("   d r_(i,j+1)(z) + T_j(z+2^i) - T_j(z)  in B_D,  based at z with z_j = 0")
for n in range(3,9):
    N, eidx, E, pairs, und, BD = build(n)
    a0 = [0,0]; a1 = [0,0]
    for (x,i,j) in und:
        if j > n-2: continue
        z = x & ~(1<<j)                      # the basepoint with seam bit 0
        d = {}
        addd(d, dsquare(n,eidx,z,i,j+1))
        addd(d, triangle(n,eidx,(z+(1<<i))%N,j))
        addd(d, triangle(n,eidx,z,j), -1)
        f = bool(np.any(BD.reduce(vec(d,E))))
        tgt = a1 if (x>>j)&1 else a0
        tgt[0] += 1; tgt[1] += f
    print(f"  n={n}:  from x_j=0 -> {a0[1]:4d}/{a0[0]:4d} fail   "
          f"from x_j=1 -> {a1[1]:4d}/{a1[0]:4d} fail   "
          f"{'OK ' if a0[1]==a1[1]==0 else 'FAIL'}")

print("\n== T_j vanishes identically at the top ==")
for n in range(3,8):
    N, eidx, elist = arena(n); E = len(elist)
    nz = sum(bool(np.any(vec(triangle(n,eidx,y,n-2),E))) for y in range(N))
    print(f"  n={n}: T_(n-2) nonzero at {nz}/{N} bases (a real relation);"
          f"  the top generator 2^(n-1) is an involution, so T_(n-1) is not even defined")

== the domino identity, exact, for x_j = 0 ==
  n=3:     4 instances, 0 failures
  n=4:    24 instances, 0 failures
  n=5:    96 instances, 0 failures
  n=6:   320 instances, 0 failures
  n=7:   960 instances, 0 failures

== the one remaining membership, split on the seam bit ==
   d r_(i,j+1)(z) + T_j(z+2^i) - T_j(z)  in B_D,  based at z with z_j = 0
  n=3:  from x_j=0 ->    0/   2 fail   from x_j=1 ->    0/   2 fail   OK 
  n=4:  from x_j=0 ->    0/  10 fail   from x_j=1 ->    0/  10 fail   OK 
  n=5:  from x_j=0 ->    0/  34 fail   from x_j=1 ->    0/  34 fail   OK 
  n=6:  from x_j=0 ->    0/  98 fail   from x_j=1 ->    0/  98 fail   OK 
  n=7:  from x_j=0 ->    0/ 258 fail   from x_j=1 ->    0/ 258 fail   OK 
  n=8:  from x_j=0 ->    0/ 642 fail   from x_j=1 ->    0/ 642 fail   OK 

== T_j vanishes identically at the top ==
  n=3: T_(n-2) nonzero at 8/8 bases (a real relation);  the top generator 2^(n-1) is an involution, so T_(n-1) is not even defined
  n=4: T_(n-2) nonzero at 16

## 16. Corrections log, v3

7. **"LSB-first is the unique minimum" has an $n=2$ exception** (a reviewer's catch; the v2
   notebook's own §4 output already printed *2 of 2 orders* at $n=2$ and the abstract
   overstated it). Correct statement: at $n=2$ both orders are optimal; for $n\ge3$ LSB-first
   is the unique optimal order, exhaustively to $n=8$.
8. **The membership test must be based at the seam-zero point.** Tested naively at $x$ it
   fails on exactly half the instances — every $x$ with $x_j=1$. That is $x\oplus2^j=x-2^j$,
   not $x+2^j$: a basepoint convention. Based correctly, zero failures to $n=8$.

## 17. Open, v3

1. **One membership statement is the whole remaining gap:**
   $\partial r_{i,j+1}(z)+T_j(z+2^i)-T_j(z)\in B_D$ for $z_j=0$, $j\le n-2$. Everything else
   in the general-$n$ theorem is now an identity or a proved lemma. A chain-level proof of
   this one finishes $b_1=\binom n2+(n-1)$ for all $n$.
2. **$\dim M_{ij}=n-j$ directly.** The normal form says pair $(i,j)$'s image is spanned by
   $\{Q_{ij}^m\}_{m=0}^{\,n-1-j}$ — exactly $n-j$ vectors, so they must be independent in the
   $c$-basis. Worth proving, and it would confirm that the module dimension *is* the number
   of carry-run depths above the seam.
3. **A general-$n$ proof of the reader law $F(S)$ itself.** Distinct from the homological
   theorem, and still finite-order verification only.
4. **Carry-lookahead readers** — prefix aggregates instead of positions. Different declared
   relations, unknown cut algebra.
5. **Arenas where memory and residual agree** — state set with group structure, no odd
   interior factors. Register rung is the trivial case.
6. **How physics selects the predicate** — untouched.

---

*v3 — September 2026. Every number above is live output from this notebook.*